# Cricbuzz API Data Fetching

Use this notebook to fetch cricket data from an API, normalize the payload, and store raw match snapshots into the SQLite database used by the Streamlit app.

In [9]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
import requests

load_dotenv()

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.db_connection import get_connection, initialize_database, store_api_match_snapshots

initialize_database()
PROJECT_ROOT

PosixPath('/home/kiran/Documents/cricbuzz_livestats')

## Configure API access

Set these values before running the fetch cell. You can either export them in your shell or assign them directly below.

In [10]:
API_URL = os.getenv('CRICBUZZ_API_URL')
API_KEY = os.getenv('CRICBUZZ_API_KEY')
API_HOST = os.getenv('CRICBUZZ_API_HOST')
HEADERS = {
    'X-API-Key': API_KEY,
    'X-API-Host': API_HOST
}

API_URL, bool(API_KEY)

('https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live', True)

## Fetch the raw payload

In [12]:
response = requests.get(API_URL, headers=HEADERS, timeout=20)

payload = response.json()

type(payload), list(payload)[:5] if isinstance(payload, dict) else len(payload)

(dict, ['message'])

## Normalize the payload into a match list

Different cricket APIs return different shapes. This helper tries the common patterns and maps them into a consistent structure for storage.

In [ ]:
def extract_matches(payload):
    if isinstance(payload, list):
        raw_matches = payload
    elif isinstance(payload, dict):
        raw_matches = (
            payload.get('matches')
            or payload.get('data')
            or payload.get('typeMatches')
            or []
        )
    else:
        raw_matches = []

    matches = []

    for item in raw_matches:
        if isinstance(item, dict) and 'seriesMatches' in item:
            for series_block in item.get('seriesMatches', []):
                wrapper = series_block.get('seriesAdWrapper', {})
                series_name = wrapper.get('seriesName', '')
                for match_wrapper in wrapper.get('matches', []):
                    info = match_wrapper.get('matchInfo', {})
                    score = match_wrapper.get('matchScore', {})
                    team1 = info.get('team1', {}).get('teamName', '')
                    team2 = info.get('team2', {}).get('teamName', '')
                    team1_score = score.get('team1Score', {}).get('inngs1', {})
                    team2_score = score.get('team2Score', {}).get('inngs1', {})
                    score_summary = ' | '.join(
                        part for part in [
                            f"{team1} {team1_score.get('runs', '')}/{team1_score.get('wickets', '')}" if team1_score else '',
                            f"{team2} {team2_score.get('runs', '')}/{team2_score.get('wickets', '')}" if team2_score else '',
                        ] if part
                    )
                    matches.append({
                        'match_id': info.get('matchId'),
                        'series': series_name,
                        'team_1': team1,
                        'team_2': team2,
                        'status': info.get('status', ''),
                        'score': score_summary,
                        'venue': info.get('venueInfo', {}).get('ground', ''),
                    })
        elif isinstance(item, dict):
            matches.append({
                'match_id': item.get('match_id') or item.get('id') or item.get('matchId'),
                'series': item.get('series') or item.get('series_name') or item.get('seriesName', ''),
                'team_1': item.get('team_1') or item.get('team1') or item.get('teamOne', ''),
                'team_2': item.get('team_2') or item.get('team2') or item.get('teamTwo', ''),
                'status': item.get('status', ''),
                'score': item.get('score') or item.get('score_summary') or '',
                'venue': item.get('venue') or item.get('ground') or '',
            })

    return matches


matches = extract_matches(payload)
len(matches)

In [ ]:
matches_df = pd.DataFrame(matches)
matches_df.head(10)

## Load the normalized matches into SQLite

This stores raw API snapshots in `api_match_snapshots` so you can audit what came from the API before transforming it into analytics tables.

In [ ]:
inserted = store_api_match_snapshots(matches, source='data_fetching_notebook')
print(f'Inserted {inserted} API match snapshots into the database.')

In [ ]:
with get_connection() as connection:
    preview = pd.read_sql_query(
        '''
        SELECT fetched_at, series_name, team_1, team_2, status, score_summary, venue
        FROM api_match_snapshots
        ORDER BY id DESC
        LIMIT 20
        ''',
        connection,
    )

preview